# 11 — Mejor tiro confirmatorio: regional HGB + CNN 2.5D

Este nodo prueba la complementariedad observada entre el experto regional del nodo 10 y la CNN **2.5D image-only** del nodo 6. Ambos usan los mismos pacientes, manifiestos de folds y semilla base.

La mezcla primaria queda congelada antes de mirar este CV5: **50% de cada probabilidad raw**. Sólo después se ajusta una temperatura de forma cross-fitted. La curva completa de pesos se presenta como diagnóstico exploratorio y no reemplaza el resultado primario.

La CNN puede recorrer hasta 500 épocas, con paciencia 80 y checkpoint por época. En cada fold externo, dos folds internos del train seleccionan `best_epoch`; posteriormente se reentrena sobre todo el train externo hasta la mediana de esas épocas. El fold externo se predice una sola vez y nunca controla el early stopping.

HGB aplica el contrato análogo: hasta 500 iteraciones, early stopping interno con paciencia 60, selección de `best_iteration` y refit sobre todo el train externo.


In [1]:
from __future__ import annotations

import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modeling.cnn.io import RunLock
from modeling.node11_confirmatory import (
    ExperimentConfig,
    blend_weight_figure,
    cnn_epoch_loss_figure,
    hgb_iteration_loss_figure,
    optuna_pairwise_scatter_figures,
    prepare_experiment,
    run_final_stage,
    run_search_stage,
)

RUN_ID = os.environ.get("DAT_NODE11_RUN_ID", "node11_best_shot_v1")
COMPLETED_TRIALS = max(10, int(os.environ.get("DAT_NODE11_COMPLETED_TRIALS", "10")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENT = ExperimentConfig(run_id=RUN_ID)
EXPERIMENT = replace(
    EXPERIMENT,
    search=replace(EXPERIMENT.search, completed_trials_per_expert=COMPLETED_TRIALS),
)
RUN_DIR = PROJECT_ROOT / "outputs" / "private_eda" / "node11_runs" / RUN_ID

display(Markdown(
    f"**Run:** `{RUN_ID}` · **device:** `{DEVICE}` · **épocas CNN máximas:** "
    f"`{EXPERIMENT.train.max_epochs}` · **patience CNN:** `{EXPERIMENT.train.cnn_patience}` · "
    f"**iteraciones HGB máximas:** `{EXPERIMENT.train.hgb_max_iter}` · "
    f"**trials COMPLETE por experto:** `{EXPERIMENT.search.effective_completed_trials}`"
))


**Run:** `node11_best_shot_v1` · **device:** `cuda` · **épocas CNN máximas:** `500` · **patience CNN:** `80` · **iteraciones HGB máximas:** `500` · **trials COMPLETE por experto:** `10`

## 1. Cohorte y folds comunes

Se reutilizan los caches físicos de los nodos 6 y 10. No se repite el registro. `acquisition_family` participa únicamente en el balance de los folds y en la auditoría posterior; nunca entra como predictor.


In [2]:
with RunLock(RUN_DIR / "prepare.lock"):
    prepared = prepare_experiment(EXPERIMENT, project_root=PROJECT_ROOT)

display(pd.DataFrame({
    "indicador": [
        "pacientes únicos", "variables regionales", "folds búsqueda comunes",
        "folds finales comunes", "trials completos por experto",
        "máximo de entrenamientos CNN en búsqueda",
    ],
    "valor": [
        prepared.cohort["uid"].nunique(),
        len([c for c in prepared.cohort if c.startswith("regional930_")]),
        EXPERIMENT.search.n_splits_search,
        EXPERIMENT.search.n_splits_final,
        EXPERIMENT.search.effective_completed_trials,
        EXPERIMENT.search.effective_completed_trials * EXPERIMENT.search.n_splits_search,
    ],
}))
display(pd.read_csv(RUN_DIR / "config" / "search_fold_summary.csv"))
display(pd.read_csv(RUN_DIR / "config" / "final_fold_summary.csv"))


,indicador,valor
0,pacientes únicos,1362
1,variables regionales,930
2,folds búsqueda comunes,3
3,folds finales comunes,5
4,trials completos por experto,10
5,máximo de entrenamientos CNN en búsqueda,30


,fold,n,pathologic,prevalence,families,background_valid_fraction
0,0,454,249,0.548458,68,0.182819
1,1,454,249,0.548458,69,0.182819
2,2,454,249,0.548458,73,0.182819


,fold,n,pathologic,prevalence,families,background_valid_fraction
0,0,273,150,0.549451,53,0.186813
1,1,273,150,0.549451,49,0.190476
2,2,272,149,0.547794,51,0.187500
3,3,272,149,0.547794,50,0.169118
4,4,272,149,0.547794,57,0.180147


## 2. Optuna prudente y reanudable

Hay sólo dos estudios. El espacio es local y se ancla en los ganadores anteriores: learning rate menor y regularización alrededor de las regiones ya prometedoras. Un trial podado no cuenta; cada estudio continúa hasta alcanzar el mínimo de trials `COMPLETE`.

SQLite conserva el estudio, y la CNN guarda `last.pt`, `last.prev.pt`, `best.pt` e historia por época para cada combinación y fold. Si la ejecución se corta, vuelve a ejecutar esta celda con el mismo `RUN_ID`.


In [3]:
with RunLock(RUN_DIR / "search.lock"):
    search_result = run_search_stage(prepared, EXPERIMENT, device=DEVICE)

display(search_result.summary)
assert (search_result.summary["completed_trials"] >= 10).all()


[I 2026-09-02 10:29:50,252] A new study created in RDB with name: node11_regional_hgb
[I 2026-09-02 10:34:01,199] Trial 0 finished with value: 0.5337563149734464 and parameters: {'learning_rate': 0.03341260125347943, 'max_leaf_nodes': 16, 'min_samples_leaf': 45, 'l2_regularization': 0.00014495975114378547}. Best is trial 0 with value: 0.5337563149734464.
[I 2026-09-02 10:37:46,348] Trial 1 finished with value: 0.5323124475606509 and parameters: {'learning_rate': 0.05001537469585738, 'max_leaf_nodes': 12, 'min_samples_leaf': 58, 'l2_regularization': 0.00010957047003794964}. Best is trial 1 with value: 0.5323124475606509.
[I 2026-09-02 10:41:52,813] Trial 2 finished with value: 0.5286064933951232 and parameters: {'learning_rate': 0.025670585689049234, 'max_leaf_nodes': 20, 'min_samples_leaf': 61, 'l2_regularization': 0.0046265451466631115}. Best is trial 2 with value: 0.5286064933951232.
[I 2026-09-02 10:45:59,820] Trial 3 finished with value: 0.5308675485656523 and parameters: {'learnin

[CNN] fold=0 epoch=1/500 train=0.6879 val_log_loss=0.6779 best=0.6779
[CNN] fold=0 epoch=2/500 train=0.6862 val_log_loss=0.6578 best=0.6578
[CNN] fold=0 epoch=3/500 train=0.6579 val_log_loss=0.6155 best=0.6155
[CNN] fold=0 epoch=4/500 train=0.6362 val_log_loss=0.5744 best=0.5744
[CNN] fold=0 epoch=5/500 train=0.5913 val_log_loss=0.6011 best=0.5744
[CNN] fold=0 epoch=6/500 train=0.5641 val_log_loss=0.5212 best=0.5212
[CNN] fold=0 epoch=7/500 train=0.5632 val_log_loss=0.5830 best=0.5212
[CNN] fold=0 epoch=8/500 train=0.5439 val_log_loss=0.5242 best=0.5212
[CNN] fold=0 epoch=9/500 train=0.5545 val_log_loss=0.5577 best=0.5212
[CNN] fold=0 epoch=10/500 train=0.5299 val_log_loss=0.5631 best=0.5212
[CNN] fold=0 epoch=11/500 train=0.5241 val_log_loss=0.6148 best=0.5212
[CNN] fold=0 epoch=12/500 train=0.5185 val_log_loss=0.5223 best=0.5212
[CNN] fold=0 epoch=13/500 train=0.5050 val_log_loss=0.6079 best=0.5212
[CNN] fold=0 epoch=14/500 train=0.5090 val_log_loss=0.6106 best=0.5212
[CNN] fold=0 ep

[W 2026-09-02 13:38:46,236] Trial 0 failed with parameters: {'base_channels': 12, 'image_embedding_dim': 128, 'pooling': 'gap_max', 'dropout': 0.360124684611175, 'learning_rate': 0.00012, 'weight_decay': 0.0010914487203784716, 'consistency_weight': 0.004944629447301126, 'noise_std': 0.021731977854449854, 'rotation_degrees': 5.743268338153312} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\modeling\node11_confirmatory\search.py", line 263, in objective
    direction="minimize",
                     ^^^^
  File "C:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\modeling\cnn\training.py", line 375, in train_one_fo

KeyboardInterrupt: 

### Regiones exploradas por Optuna

Cada matriz contiene todos los pares de hiperparámetros del estudio. Cada punto es un trial completo y el color representa su log loss CV3. Para learning rate y regularización se muestra `log10`, evitando que los puntos queden visualmente comprimidos.


In [ ]:
for study_name, figure in optuna_pairwise_scatter_figures(RUN_DIR).items():
    display(Markdown(f"#### `{study_name}`"))
    figure.show()


## 3. CV5 final sin usar el fold externo para early stopping

Para cada fold final, el HGB y la CNN reciben exactamente los mismos UIDs de train y validación. La selección de época/iteración ocurre sólo dentro del train. El resultado guarda los dos expertos, la mezcla primaria 50/50, calibración, bootstrap pareado, peor familia de adquisición y una curva exploratoria de pesos.

Esta etapa también es reanudable. La CNN puede tardar: por fold se ejecutan dos trayectorias internas de hasta 500 épocas y un refit hasta el `best_epoch` robusto.


In [ ]:
with RunLock(RUN_DIR / "final.lock"):
    final_result = run_final_stage(prepared, EXPERIMENT, device=DEVICE)

display(final_result.metrics)
display(final_result.bootstrap_differences)
display(pd.read_csv(RUN_DIR / "final" / "cnn_epoch_selection.csv"))
display(pd.read_csv(RUN_DIR / "final" / "hgb_iteration_selection.csv"))
display(Markdown(
    f"**Primario preespecificado:** `{final_result.deployment_manifest['primary_family']}` · "
    f"**folds comunes:** `{final_result.deployment_manifest['n_outer_folds']}` · "
    f"**fold externo usado para early stopping:** "
    f"`{final_result.deployment_manifest['outer_fold_used_for_early_stopping']}`"
))


## 4. Época versus loss: diagnóstico de sobreajuste

Las líneas sólidas son log loss de validación **interna** y las punteadas son la pérdida de entrenamiento. Las estrellas indican los `best_epoch` usados para decidir cuántas épocas tendrá el refit de cada fold externo.

Una separación creciente entre train y validación, acompañada de una validación que empeora, indica sobreajuste. Si ambas siguen descendiendo al detenerse, la paciencia u horizonte podrían seguir siendo insuficientes.


In [ ]:
cnn_epoch_loss_figure(RUN_DIR).show()


## 5. Iteración versus loss del regional HGB


In [ ]:
hgb_iteration_loss_figure(RUN_DIR).show()


## 6. Complementariedad final

El diamante 50/50 es el resultado primario porque su peso fue congelado antes del CV5. La estrella marca el mínimo retrospectivo de la curva y sirve sólo para formular el siguiente experimento; no debe reportarse como estimación imparcial del desempeño.


In [ ]:
blend_weight_figure(RUN_DIR).show()
display(pd.read_json(RUN_DIR / "final" / "complementarity.json", typ="series"))
display(pd.read_csv(RUN_DIR / "final" / "subgroup_metrics_by_acquisition_family.csv"))


## Criterio de promoción

Promover la mezcla sólo si mejora el log loss OOF frente al HGB regional, el bootstrap pareado favorece la mezcla, no empeora sustancialmente la peor familia soportada y las curvas internas no muestran inestabilidad grave. El leaderboard sigue siendo validación externa adicional, no sustituto del CV.
